## 0. Setup

# Review Queue: Items Most in Need of Human Review

`RareBooks_DataExtraction.ipynb` flags individual extracted items with `review_flags` whenever the LLM had to guess at a garbled OCR reading, infer a field from context, or make some other judgment call. This notebook turns those flags into a single ranked **CSV review queue**: for each flagged item, what the OCR/verbatim text actually said, how the LLM resolved it into a structured field, why it was flagged, and exactly where to find it in the source PDF.

- **Source:** `Structured Data/concert_program_items.json` — the flat item export (one row per venue/date/organization/patron/work/performer).
- **Not used:** `Structured Data/langchain_flags.csv`. It's a flag log from a *different* extraction pass (`code/Langchain_for_Rare_Books.ipynb`, over a different, non-concert-program source) and doesn't carry a source filename per flag — only a page number — so there's no reliable way to give a "source document" for each row. If that pipeline's output ever gets a filename column, it can be folded in here the same way.


In [2]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px

data_dir = Path("Structured Data")

with open(data_dir / "concert_program_items.json", encoding="utf-8") as f:
    items = json.load(f)

print(f"{len(items)} total items")
flagged = [x for x in items if x.get("review_flags")]
print(f"{len(flagged)} items carry at least one review flag ({len(flagged) / len(items):.0%})")


11379 total items
1983 items carry at least one review flag (17%)


## 1. Turn each flagged item into a review-queue row

Two judgment calls happen here:

- **LLM Resolution** — each `record_type` keeps different fields (a `work` has `composer`/`title`/`movement_or_selection`; a `performer` has `name`/`part_or_instrument`/`associated_work`; ...). `resolution_text()` renders just the meaningful fields for that record type into one readable string, so the CSV can be scanned without knowing the schema.
- **Severity** — flags aren't equally urgent. A flag whose text mentions OCR garbling, illegibility, or uncertainty means the *reading itself* is in doubt; a flag like "Role standardized to 'actor' from 'cast' wording" documents a clean judgment call, not a data-quality problem. `classify_severity()` buckets flags into:
  - **High — OCR/legibility**: the source text may have been misread (`OCR`, `garble`, `illegible`, `unclear`, `misread`, `uncertain`, `faded`, `unreadable`, `cut off`)
  - **Medium — inference**: the text was legible but the LLM had to infer/standardize a field (`inferred`, `assumed`, `implied`, `cross-page`, `best-guess`, `standardized`, `not specified`, `not stated`, `not given`, `missing`)
  - **Low — informational**: a flag documenting an intentional schema choice, not a likely error

An item with multiple flags takes its *highest* severity.


In [3]:
RESOLUTION_FIELDS = {
    "venue": ["place"],
    "date": ["date_text"],
    "organization": ["name", "role"],
    "patron": ["name", "role"],
    "work": ["title", "composer", "movement_or_selection"],
    "performer": ["name", "part_or_instrument", "associated_work"],
}

def resolution_text(item):
    """Render the meaningful structured fields for this record_type as one readable string."""
    fields = RESOLUTION_FIELDS.get(item["record_type"], [])
    parts = [f"{field}: {item[field]}" for field in fields if item.get(field)]
    return " | ".join(parts)

HIGH_KEYWORDS = [
    "ocr", "garble", "illegible", "unclear", "misread", "uncertain",
    "faded", "unreadable", "cut off", "unrecoverable",
]
MEDIUM_KEYWORDS = [
    "inferred", "assumed", "implied", "cross-page", "best-guess", "best guess",
    "standardized", "not specified", "not stated", "not given", "missing", "likely",
]

def classify_severity(flags):
    """Highest severity across an item's flags: High (2) > Medium (1) > Low (0)."""
    score = 0
    for flag in flags:
        fl = flag.lower()
        if any(kw in fl for kw in HIGH_KEYWORDS):
            score = max(score, 2)
        elif any(kw in fl for kw in MEDIUM_KEYWORDS):
            score = max(score, 1)
    return score

SEVERITY_LABELS = {2: "High — OCR/legibility", 1: "Medium — inference", 0: "Low — informational"}


In [4]:
rows = []
for item in flagged:
    score = classify_severity(item["review_flags"])
    rows.append({
        "Severity": SEVERITY_LABELS[score],
        "_severity_score": score,
        "Record Type": item["record_type"],
        "Source Document": item["filename"],
        "Page": item["page_number"],
        "Organization": item.get("manifest_organization"),
        "Date": item.get("manifest_date"),
        "OCR Original": item.get("source_text"),
        "LLM Resolution": resolution_text(item),
        "Suggested Problem": "; ".join(item["review_flags"]),
    })

review_queue = pd.DataFrame(rows).sort_values(
    ["_severity_score", "Source Document", "Page"], ascending=[False, True, True]
)
print(f"{len(review_queue)} rows in the review queue")
review_queue.drop(columns="_severity_score").head(15)


1983 rows in the review queue


,Severity,Record Type,Source Document,Page,Organization,Date,OCR Original,LLM Resolution,Suggested Problem
70,High — OCR/legibility,organization,UDC20260028-1.pdf,1,Musashinto Academia Musicae,1967-68,MUSASHINO ACADEMIA MUSICAЕ,name: MUSASHINO ACADEMIA MUSICAЕ | role: prese...,OCR shows 'MUSICAЕ'; likely 'MUSICAE' (Cyrilli...
74,High — OCR/legibility,work,UDC20260028-1.pdf,20,Musashinto Academia Musicae,1967-68,N. K. Medtner ...................................,"title: ""Four Fairy Tales"" Op. 26 | composer: N...",OCR garble around quotes; title recovered from...
14,High — OCR/legibility,performer,UDC20260028-10.pdf,5,Melbourne Liedertafel,1893,"Marshall, Capt. T. S. (on l.","name: Marshall, Capt. T. S. (on l. | part_or_i...",OCR garble in '(on leave)'. Marked as on leave...
16,High — OCR/legibility,performer,UDC20260028-10.pdf,5,Melbourne Liedertafel,1893,"Morgan, C. W. on leave)","name: Morgan, C. W. on leave) | part_or_instru...",OCR punctuation garble; marked as on leave; ma...
28,High — OCR/legibility,organization,UDC20260028-11.pdf,1,Melbourne Liedertafel,1893,THE ROYAL METROPOLITAN LIEGEBRIGADE\nR.M.L.,name: THE ROYAL METROPOLITAN LIEGEBRIGADE | ro...,Name appears garbled by OCR; likely intended t...
46,High — OCR/legibility,performer,UDC20260028-11.pdf,5,Melbourne Liedertafel,1893,J. B. Pewtrriss .. 1892,name: J. B. Pewtrriss | part_or_instrument: tenor,Surname may be an OCR error
47,High — OCR/legibility,performer,UDC20260028-11.pdf,5,Melbourne Liedertafel,1893,J. Fordvce .. 1872,name: J. Fordvce | part_or_instrument: bass,Surname may be an OCR error
30,High — OCR/legibility,work,UDC20260028-11.pdf,9,Melbourne Liedertafel,1893,"Part Song—""The Miller's Daughter"" Hartel",title: The Miller's Daughter | composer: Hartel,Composer credit 'Hartel' may refer to publishe...
31,High — OCR/legibility,work,UDC20260028-11.pdf,11,Melbourne Liedertafel,1893,"Song—""Alla Stella Confidente"" Bobaudi\n(With C...",title: Alla Stella Confidente | composer: Bobaudi,Composer name may be an OCR error
32,High — OCR/legibility,work,UDC20260028-11.pdf,11,Melbourne Liedertafel,1893,"Part Song—""The Mariner's Return"" Höcsler",title: The Mariner's Return | composer: Höcsler,Composer name may be an OCR error


In [6]:
# selected document for review
review_queue[review_queue['Source Document']=='UDC20260028-21.pdf']

,Severity,_severity_score,Record Type,Source Document,Page,Organization,Date,OCR Original,LLM Resolution,Suggested Problem
146,High — OCR/legibility,2,organization,UDC20260028-21.pdf,3,Melbourne Liedertafel,1899,MELBOURNE @ LIEDERTAFEL.,name: MELBOURNE @ LIEDERTAFEL. | role: present...,'@' likely OCR for '&'
229,High — OCR/legibility,2,performer,UDC20260028-21.pdf,5,Melbourne Liedertafel,1899,Burton J. W. (on leave),name: Burton J. W. (on leave) | part_or_instru...,Part inferred from 'FIRST TENOR.' header; OCR ...
236,High — OCR/legibility,2,performer,UDC20260028-21.pdf,5,Melbourne Liedertafel,1899,"Blundell, K. S. P (on I've)","name: Blundell, K. S. P (on I've) | part_or_in...",Part inferred from 'SECOND TENOR.' header; 'on...
252,High — OCR/legibility,2,performer,UDC20260028-21.pdf,5,Melbourne Liedertafel,1899,"Muirhead, G. H. (on I've)","name: Muirhead, G. H. (on I've) | part_or_inst...",Part inferred from 'FIRST BASS.' header; 'on I...
263,High — OCR/legibility,2,performer,UDC20260028-21.pdf,5,Melbourne Liedertafel,1899,"Brodrribb, H. B.","name: Brodrribb, H. B. | part_or_instrument: f...",Part inferred from 'FIRST BASS.' header; Possi...
...,...,...,...,...,...,...,...,...,...,...
147,Medium — inference,1,patron,UDC20260028-21.pdf,18,Melbourne Liedertafel,1899,His Excellency the Governor and Lady Brassey h...,name: His Excellency the Governor | role: Patron,Assumed 'His Excellency the Governor' refers t...
205,Low — informational,0,performer,UDC20260028-21.pdf,5,Melbourne Liedertafel,1899,FIRST TENOR.,"name: Dickins, J. J., Capt. | part_or_instrume...",Part applied to following names until next sec...
233,Low — informational,0,performer,UDC20260028-21.pdf,5,Melbourne Liedertafel,1899,SECOND TENOR.,"name: Hall, A. G., Capt. | part_or_instrument:...",Part applied to following names until next sec...
251,Low — informational,0,performer,UDC20260028-21.pdf,5,Melbourne Liedertafel,1899,FIRST BASS.,"name: Rendle, S. Capt. | part_or_instrument: f...",Part applied to following names until next sec...


## 2. Quick shape check before exporting

In [7]:
fig = px.bar(
    review_queue.groupby(["Severity", "Record Type"]).size().reset_index(name="count"),
    x="Severity",
    y="count",
    color="Record Type",
    title="Flagged items by severity and record type",
    category_orders={"Severity": ["High — OCR/legibility", "Medium — inference", "Low — informational"]},
)
fig.update_layout(height=500)
fig.show()


In [8]:
fig = px.bar(
    review_queue.groupby("Organization").size().reset_index(name="flagged_items").sort_values("flagged_items", ascending=False),
    x="flagged_items",
    y="Organization",
    orientation="h",
    title="Flagged items by organization",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=400)
fig.show()


## 3. Export the CSV

Sorted worst-first (High severity → Medium → Low, then by document/page) so a human reviewer can work down the list and stop whenever they've covered the material that matters most.


In [9]:
out_path = data_dir / "review_queue.csv"
review_queue.drop(columns="_severity_score").to_csv(out_path, index=False)
print(f"Wrote {len(review_queue)} rows to {out_path}")


Wrote 1983 rows to Structured Data/review_queue.csv
